In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 2
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.478842960079122

Trial 1 =========================================
14.266583627893777

Trial 2 =========================================
13.22703386882316

Trial 3 =========================================
15.396853211074893

Trial 4 =========================================
14.304713902355898

Trial 5 =========================================
13.857204523689399

Trial 6 =========================================
13.751669612780086

Trial 7 =========================================
16.2433914097254

Trial 8 =========================================
14.637258054225644

Trial 9 =========================================
14.474237495863985

Trial 10 =========================================
14.451694707356989

Trial 11 =========================================
15.127127119292837

Trial 12 =========================================
14.276864106535367

Trial 13 =========================================
15.389553460249974

Trial 14 ==========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 26 =========================================
15.929222010399386

Trial 27 =========================================
15.442653936154786

Trial 28 =========================================
14.21288870361813

Trial 29 =========================================
18.813939467611423

Trial 30 =========================================
15.23903518274187

Trial 31 =========================================
14.264415339962131

Trial 32 =========================================
14.79895867965091

Trial 33 =========================================
13.455017386311493

Trial 34 =========================================
18.804650031658408

Trial 35 =========================================
13.423668604116067

Trial 36 =========================================
14.281129472437993

Trial 37 =========================================
17.540148746622375

Trial 38 =========================================
16.16179308542882

Trial 39 =========================================
16.157718933336636

Trial 40 =

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 44 =========================================
18.796658807283347



...

Trial 88 =========================================
15.929222010399386

Trial 89 =========================================
15.346304042851251

Trial 90 =========================================
13.144441239431584

Trial 91 =========================================
16.24123709525401

Trial 92 =========================================
18.80738973708178



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 93 =========================================
18.801535697886823

Trial 94 =========================================
14.443027709511387

Trial 95 =========================================
16.25379817676045

Trial 96 =========================================
15.104113798865516

Trial 97 =========================================
14.512034397058313



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 98 =========================================
15.676518922290889

Trial 99 =========================================
16.489433690106736

[14.47884296 14.26658363 13.22703387 15.39685321 14.3047139  13.85720452
 13.75166961 16.24339141 14.63725805 14.4742375  14.45169471 15.12712712
 14.27686411 15.38955346 14.44095749 15.92922201 18.67347115 14.77974435
 13.14004489 14.1739302  13.37528304 16.5228968  14.48396259 13.45809184
 15.39056838 14.61134504 15.92922201 15.44265394 14.2128887  18.81393947
 15.23903518 14.26441534 14.79895868 13.45501739 18.80465003 13.4236686
 14.28112947 17.54014875 16.16179309 16.15771893 13.97100939 14.35854313
 14.81204808 15.38153622 18.79665881 15.89227126 16.23849271 14.02391222
 16.06127529 15.04742619 16.22973506 13.91568918 15.98903852 15.39913285
 15.95686467 16.20215121 15.39492681 18.2959381  13.43292355 14.43667468
 15.35944913 13.12814922 17.27493104 16.13890288 17.50343005 15.66574787
 14.37186581 17.1859992  15.63835783 13.44881415 14.3034

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.813939467611423
Avg = 15.389569104899877
Std = 1.5123577122292395


In [6]:
print(y_max_arr.tolist())

[14.478842960079122, 14.266583627893777, 13.22703386882316, 15.396853211074893, 14.304713902355898, 13.857204523689399, 13.751669612780086, 16.2433914097254, 14.637258054225644, 14.474237495863985, 14.451694707356989, 15.127127119292837, 14.276864106535367, 15.389553460249974, 14.440957486241373, 15.929222010399386, 18.673471146080686, 14.779744353042403, 13.140044888363997, 14.173930200115484, 13.375283039875779, 16.52289680341676, 14.483962594416475, 13.458091838627208, 15.390568376489428, 14.611345042097156, 15.929222010399386, 15.442653936154786, 14.21288870361813, 18.813939467611423, 15.23903518274187, 14.264415339962131, 14.79895867965091, 13.455017386311493, 18.804650031658408, 13.423668604116067, 14.281129472437993, 17.540148746622375, 16.16179308542882, 16.157718933336636, 13.971009394536733, 14.358543134928635, 14.81204808164398, 15.381536216806825, 18.796658807283347, 15.892271256246492, 16.23849270561151, 14.023912221892843, 16.061275285911208, 15.047426185487055, 16.229735

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-2.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)